# Cleaning Data
v.ekc

Raw data is almost never ready to use. Today we clean the YouTube dataset from last week and learn to reshape wide/long data — two skills you'll use constantly in the wild.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Cleaning the Data |
| 3 | Reshaping Data |
| 4 | 🔬 Activity — Pets DataFrame |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# import the youtube data from last time
df = pd.read_csv('humboldt_youtube_2025.csv', dtype={'viewCount': 'object', 'likeCount':'object'})
df.head()

---
## 2. Cleaning the Data

We'll work through four common cleaning steps: fixing column names, correcting data types, setting the index, and reordering.

### 📋 Board Reference

| Task | Code | Notes |
|------|------|-------|
| Rename specific columns | `df.rename(columns={'old':'new'})` | Returns new df unless `inplace=True` |
| Rename all columns | `df.columns = [...]` | Replaces all names at once |
| Drop column | `df.drop(columns='col', inplace=True)` | |
| Remove constant columns | `df.loc[:, df.nunique() > 1]` | Keep only columns with >1 unique value |
| Convert type | `df.astype({'col': int})` | |
| Parse datetime | `pd.to_datetime(df['col'])` | Use `format=` for non-standard strings |
| Set index | `df.set_index('col', inplace=True)` | |
| Sort rows | `df.sort_values(by='col', ascending=False)` | |
| Sort columns | `df.sort_index(axis=1)` | |

### Column Names

In [ ]:
# Check the column names
df.columns

In [ ]:
# Inspect some questions
sum(df.publishedAt != df.publishTime)

In [ ]:
# the two columns above are identical
df.drop(columns='publishTime',inplace=True)

In [ ]:
# Check for any columns with just one value
df.nunique() == 1

In [ ]:
# Save some relevant data, then remove redundant columns
channel_id = df.channelId[0]
chanel_title = df.channelTitle[0]

In [ ]:
df = df.loc[:,df.nunique() > 1]

In [ ]:
df.head()

Our column names are in different formats — let's standardize them.

In [ ]:
# OPTION 1 .rename method
df.rename(columns = {'videoId':'video_id',
                     'publishedAt':'published_at',
                     'viewCount':'view_count',
                     'likeCount':'like_count'}).head()

In [ ]:
# does not update original (use inline=True)
df.head()

In [ ]:
# OPTION 2: more general approach that would work for many columns
# Split column names by words
import re
split_by_words = [re.split('(?=[A-Z])',i) for i in df.columns]
split_by_words

In [ ]:
# Insert an underscore between words
columns_with_underscores = ['_'.join(i).lower() for i in split_by_words]
columns_with_underscores

In [ ]:
# Reassign column names
df.columns = columns_with_underscores

In [ ]:
df.head()

### Convert Datatypes

In [ ]:
# Suppose we wanted to look at the relationship between view counts and like counts
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')
sns.scatterplot(data = df, x='view_count',y='like_count')
plt.show()

In [ ]:
# Something is funky. Check the data types
df.dtypes

In [ ]:
# convert the data types
df = df.astype({'view_count':int, 'like_count':int})

In [ ]:
# check data types again
df.dtypes

In [ ]:
# plot again
sns.scatterplot(data = df, x='view_count',y='like_count')
plt.show()

In [ ]:
# What if we wanted to plot trend in view counts?
sns.lineplot(data = df, x='published_at',y='view_count')
plt.show()

In [ ]:
# change dates to datetime data
df['published_at'] = pd.to_datetime(df.published_at)
df.head()

In [ ]:
# Check dtypes again
df.dtypes

In [ ]:
# plot again
plt.figure(figsize = (10,4))
sns.lineplot(data = df, x='published_at',y='view_count')
plt.show()

Pandas datetime types can be interpreted in multiple formats:

In [ ]:
date1 = 'April 5, 2025'
pd.to_datetime(date1, format = '%B %d, %Y')

In [ ]:
date2 = 'Apr 5, 2025'
pd.to_datetime(date2, format = '%b %d, %Y')

In [ ]:
date3 = '4/5/25'
pd.to_datetime(date3, format = '%m/%d/%y')

In [ ]:
date4 = '5-4-25'
pd.to_datetime(date4, format = '%d-%m-%y')

In [ ]:
date5 = '4-5-2025'
pd.to_datetime(date5, format = '%m-%d-%Y')

In [ ]:
date6 = '4-05-2025'
pd.to_datetime(date5, format = '%m-%d-%Y')

Use [this reference](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior) for more info about datetime formats.

### Reset the Index

In [ ]:
# Reset the index for easy access by video id
df.set_index('video_id',inplace=True)
df.head()

In [ ]:
# index by video id
video_id = "PoGnLtH1UHY"
df.loc[video_id]

In [ ]:
df.loc[df.view_count.idxmax()]

### Reorder the Data

In [ ]:
# reorder rows
df.sort_index(axis=0).head()

In [ ]:
# doesn't update original
df.head()

In [ ]:
# reorder columns
df.sort_index(axis=1).head()

In [ ]:
# Place columns in your chosen order
df[['view_count','like_count','published_at','title']].head()

In [ ]:
# sort the data 
df.sort_values(by='view_count').head()

In [ ]:
# doesn't update original
df.head()

In [ ]:
# do it inplace
df.sort_values(by = 'view_count',ascending=False, inplace=True)

In [ ]:
df.head()

In [ ]:
# sort by multiple values at once
df.sort_values(by = ['view_count','like_count'],ascending=[False,True], inplace=True)
df.head()

---
## 3. Reshaping Data

Data often arrives in the wrong shape for the analysis you want. Two common transformations: **wide → long** and **long → wide**.

### 📋 Board Reference

| Direction | Function | When to use |
|-----------|----------|-------------|
| Wide → Long | `.melt(id_vars=, value_vars=)` | Each observation → its own row |
| Wide → Long | `.stack()` | Column names become a new level of the index |
| Long → Wide | `.pivot(index=, columns=, values=)` | One value per cell — no aggregation |
| Long → Wide | `.pivot_table(aggfunc=)` | Multiple values per cell — aggregate them |

In [ ]:
df_weather_wide = pd.read_csv('sample_weather.csv')
df_weather_wide = df_weather_wide.iloc[:,1:]
df_weather_wide

In [ ]:
# transpose with more informative columns
date_index = df_weather_wide.set_index('date')
date_index

In [ ]:
# Make a long series
long_df = date_index.stack()
long_df

In [ ]:
long_df = long_df.reset_index(name='value')
long_df

In [ ]:
long_df.rename(columns = {'level_1':'variable'},inplace=True)
long_df

In [ ]:
# If our columns had a name
date_index.columns.name = 'variable'

In [ ]:
date_index

In [ ]:
# Now renaming isn't necessary
date_index.stack().reset_index(name = 'value')

In [ ]:
# another way
df_weather_wide

In [ ]:
# another way
long_df = df_weather_wide.melt(id_vars = 'date',value_vars = ['max_temp','min_temp','inches_of_rain'])
long_df

In [ ]:
# change long format back into wide format
long_df.pivot(index = 'date',columns = 'variable',values='value')

What do when there are multiple values in categories:

In [ ]:
long_df = pd.read_csv('long_data.csv')
long_df = long_df.iloc[:,1:]
long_df.head()

In [ ]:
# Pivot the data to get average sales by date and category
long_df.pivot_table(index=['date'], columns='category', values=['sales'])

In [ ]:
# Pivot the data to get TOTAL sales by date and category
wide_df = long_df.pivot_table(index=['date'], columns='category', values=['sales'], aggfunc=sum)
wide_df

In [ ]:
# Pivot the data to get TOTAL sales by date, product, and category
long_df.pivot_table(index='date', columns=['category','product'], values=['sales'], aggfunc=sum)

In [ ]:
# Go from wide to long
wide_df.reset_index().melt(id_vars='date', var_name=['type','category'])

---
## 4. 🔬 Activity — Pets DataFrame

Practice all three cleaning steps on a new dataset.

In [ ]:

# Create a DataFrame with data cleaning and reshaping opportunities
data = {
    'Pet Name': ['Fluffy', 'Whiskers', 'Bubbles', 'Spike', 'Coco', 'Maybelle', 'Snowball'],
    'Date Adopted': ['10-01-2023','03-04-2024','01-10-2024','02-14-2024','11-22-2023','01-04-2024','12-25-2025'],
    'Animal Type': ['Cat', 'Cat', 'Fish', 'Dog', 'Fish', 'Dog', 'Cat'],
    'Pet Age': ['3', '2', '13', '5', '4', '3', '2'],
    'Color': ['White', 'Gray', 'Orange', 'White', 'White', 'Black', 'Black'],
    'Happiness Level': ['High', 'Medium', 'High', 'Low', 'High', 'High', 'Medium']
}
df_pets = pd.DataFrame(data)
df_pets

---
### Try It 1: Clean the Pets DataFrame 😊

1. **Rename columns** to a consistent format (e.g., `snake_case` or `lowercase with spaces` — your choice).

2. **Fix datatypes** — any columns that should be converted? Think about `Pet Age` and `Date Adopted`.

3. **Reshape** — use `.pivot_table()` to find the average age by `Animal Type`. Then try melting the result back to long format.

In [ ]:
# 1. Rename columns


In [ ]:
# 2. Fix datatypes


In [ ]:
# 3. pivot_table: average age by Animal Type


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Many column naming conventions work — consistency is what matters.*

```python
# 1. Rename
df_pets = df_pets.rename(columns={
    'Pet Name': 'pet_name',
    'Date Adopted': 'date_adopted',
    'Animal Type': 'animal_type',
    'Pet Age': 'pet_age',
    'Color': 'color',
    'Happiness Level': 'happiness_level'
})

# 2. Fix types
df_pets['pet_age'] = df_pets['pet_age'].astype(int)
df_pets['date_adopted'] = pd.to_datetime(df_pets['date_adopted'], format='%m-%d-%Y')

# 3. pivot_table
df_pets.pivot_table(index='animal_type', values='pet_age', aggfunc='mean')
```

</details>

---
## Appendix — Cleaning & Reshaping Quick Reference

### Cleaning
```python
df.rename(columns={'old':'new'}, inplace=True)       # rename specific columns
df.columns = ['col1', 'col2', ...]                   # rename all columns
df.drop(columns='col', inplace=True)                 # drop a column
df = df.loc[:, df.nunique() > 1]                     # drop constant columns
df = df.astype({'col': int})                         # convert type
df['col'] = pd.to_datetime(df['col'])                # parse datetime
df.set_index('col', inplace=True)                    # set index
df.sort_values(by='col', ascending=False, inplace=True)  # sort rows
```

### Reshaping
```python
# Wide → Long
df.melt(id_vars='id_col', value_vars=['a','b'])       # melt
df.set_index('id').stack().reset_index(name='value')  # stack

# Long → Wide
df.pivot(index='id', columns='var', values='val')     # pivot (no aggregation)
df.pivot_table(index='id', columns='var',             # pivot_table (aggregates)
               values='val', aggfunc=sum)
```